# 06 — Training: Swin-S (Local — RTX 5060 Ti 16 GB)

Entrenamiento de **Swin Transformer Small (Swin-S)** preentrenado en ImageNet para clasificación morfológica de galaxias (6 clases).  
Versión local optimizada para **NVIDIA RTX 5060 Ti 16 GB** (Blackwell, sm_120).

| Hiperparámetro | Valor |
|---|---|
| Batch size | 32 (16 GB VRAM) — reducir a 16 si tienes 8 GB |
| LR backbone | 1e-4 |
| LR head | 1e-3 |
| Optimizer | AdamW (wd=1e-4) |
| Scheduler | CosineAnnealingLR (T_max=30) |
| Epochs | 30 (+ early stopping, paciencia=5) |
| AMP | ✅ float16 |
| torch.compile | ✅ Linux / ❌ Windows (sin Triton) |
| Early stopping | ✅ paciencia=5 epochs sin mejora en val F1 |
| IMAGE_SIZE | 308 px (= 11 × 28 — divisible por patch_size × window_size) |

> **¿Por qué Swin-S y no Swin-T?**  
> Swin-S tiene los mismos hiperparámetros que T (embed_dim=96, window_size=7) pero con 18 bloques en el stage 3 en lugar de 6.  
> Esto le permite capturar dependencias espaciales más largas — crucial para distinguir Elliptical de Lenticular.  
> Con 16 GB VRAM el consumo estimado (~5-6 GB a BS=32) es muy holgado.

> **Requisito de resolución:** IMAGE_SIZE debe ser divisible por `patch_size × window_size = 4 × 7 = 28`.  
> 308 = 11 × 28 ✓ — el bias posicional relativo en cada ventana 7×7 transfiere perfectamente desde ImageNet.

> **Requisito de CUDA:** RTX 5060 Ti requiere **CUDA 12.8+** y **PyTorch ≥ 2.7**. Ejecuta primero la celda de instalación.

## Sección 0 — Instalación de dependencias

Ejecuta esta celda **una sola vez**. Reinicia el kernel después si es la primera instalación.

In [ ]:
import subprocess, sys

# PyTorch con CUDA 12.8 (necesario para RTX 5060 Ti / Blackwell sm_120)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu128',
    '--quiet',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet',
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'Pillow', 'scikit-learn', 'tqdm',
], check=True)

print('Instalación completada. Reinicia el kernel si es la primera vez.')

## Sección 1 — Imports

In [ ]:
import gc
import os
import sys
import time
import pathlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import swin_s, Swin_S_Weights
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    cap = torch.cuda.get_device_capability(0)
    print(f'Compute  : sm_{cap[0]}{cap[1]}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')

## Sección 2 — Configuración

In [ ]:
# Paths locales
_LOCAL     = pathlib.Path('../data')
IMAGES_DIR = _LOCAL / 'images_gz2' / 'images'
SPLITS_DIR = _LOCAL / 'splits'
CKPT_DIR   = pathlib.Path('../models/checkpoints/swin_s')
LOG_DIR    = pathlib.Path('../logs')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGES_DIR.exists(), f'No se encontró IMAGES_DIR: {IMAGES_DIR.resolve()}'
assert (SPLITS_DIR / 'train.csv').exists(), f'No se encontró train.csv en {SPLITS_DIR.resolve()}'

# Model
MODEL_NAME   = 'swin_s'
NUM_CLASSES  = 6
CLASS_ORDER  = ['Elliptical', 'Lenticular', 'Spiral', 'Barred_Spiral', 'Edge_on', 'Irregular']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# Training
EPOCHS         = 30
EARLY_STOP_PAT = 5      # epochs sin mejora en val F1 antes de detener
BATCH_SIZE     = 128     # 16 GB VRAM con AMP (~5-6 GB); reducir a 16 en 8 GB

# En Windows + Jupyter, num_workers > 0 causa deadlock (workers reimportan el kernel).
NUM_WORKERS    = 0 if os.name == 'nt' else 4

LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP      = device.type == 'cuda'
# torch.compile() requiere Triton — solo disponible en Linux/macOS.
USE_COMPILE  = os.name != 'nt'

# Swin requiere IMAGE_SIZE divisible por patch_size × window_size = 4 × 7 = 28.
# 308 = 11 × 28 ✓.  CenterCrop(380) elimina el borde negro de las imágenes GZ2 (424×424).
IMAGE_SIZE    = 308
CROP_SIZE     = 380
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RANDOM_SEED   = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'OS             : {"Windows" if os.name == "nt" else "Linux/macOS"}')
print(f'IMAGES_DIR     : {IMAGES_DIR.resolve()}')
print(f'SPLITS_DIR     : {SPLITS_DIR.resolve()}')
print(f'CKPT_DIR       : {CKPT_DIR.resolve()}')
print(f'MODEL          : {MODEL_NAME}')
print(f'EPOCHS         : {EPOCHS}  (early stop paciencia={EARLY_STOP_PAT})')
print(f'BATCH_SIZE     : {BATCH_SIZE}')
print(f'IMAGE_SIZE     : {IMAGE_SIZE}  ({IMAGE_SIZE} = {IMAGE_SIZE // 28} × 28 ✓)')
print(f'CROP_SIZE      : {CROP_SIZE}')
print(f'NUM_WORKERS    : {NUM_WORKERS}')
print(f'LR backbone    : {LR_BACKBONE}')
print(f'LR head        : {LR_HEAD}')
print(f'USE_AMP        : {USE_AMP}')
print(f'USE_COMPILE    : {USE_COMPILE}')

## Sección 3 — Pipeline de datos

In [ ]:
train_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class GalaxyDataset(Dataset):
    """Almacena numpy arrays para evitar copy-on-read en workers forked."""
    def __init__(self, csv_path, images_dir, transform, class_to_idx):
        df = pd.read_csv(
            csv_path,
            usecols=['img_filename', 'morph_label'],
            dtype={'img_filename': 'str', 'morph_label': 'str'},
        )
        self.filenames = df['img_filename'].to_numpy()
        self.labels    = np.array(
            [class_to_idx[lbl] for lbl in df['morph_label']], dtype=np.int64
        )
        del df
        gc.collect()
        self.images_dir = pathlib.Path(images_dir)
        self.transform  = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image = Image.open(self.images_dir / self.filenames[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])


g = torch.Generator().manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'train.csv', IMAGES_DIR, train_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'), generator=g,
)
val_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'val.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

print(f'Train batches : {len(train_loader):,}  ({len(train_loader.dataset):,} imgs)')
print(f'Val   batches : {len(val_loader):,}  ({len(val_loader.dataset):,} imgs)')

In [ ]:
# Class weights para CrossEntropyLoss
_df_w = pd.read_csv(
    SPLITS_DIR / 'train.csv',
    usecols=['morph_label'],
    dtype={'morph_label': 'str'},
)
label_counts  = np.bincount(_df_w['morph_label'].map(CLASS_TO_IDX).values, minlength=NUM_CLASSES)
weights       = len(_df_w) / (NUM_CLASSES * label_counts)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print('Class weights:')
for cls, w, n in zip(CLASS_ORDER, weights, label_counts):
    print(f'  {cls:<15}  n={n:>6,}   w={w:.4f}')

del _df_w
gc.collect()
print('RAM liberada ✓')

## Sección 4 — Modelo Swin-S

`swin_s` de torchvision usa:
- `embed_dim = 96`, `window_size = 7`, `patch_size = 4`
- Profundidades por stage: `[2, 2, 18, 2]`  (vs `[2, 2, 6, 2]` en Swin-T)
- Cabeza: `Linear(768, 1000)` → sustituimos por `Linear(768, 6)`
- Dimensiones de los bloques: 96 → 192 → 384 → 768  (= 96 × 2^stage_idx)

In [ ]:
base_model = swin_s(weights=Swin_S_Weights.IMAGENET1K_V1)

# Reemplaza la cabeza clasificadora (Linear(768, 1000) → Linear(768, NUM_CLASSES))
in_features = base_model.head.in_features
base_model.head = nn.Linear(in_features, NUM_CLASSES)
print(f'Head   : Linear({in_features}, {NUM_CLASSES})')
print(f'Params : {sum(p.numel() for p in base_model.parameters()):,}')

# Grupos de parámetros — ANTES de torch.compile()
backbone_params = [p for n, p in base_model.named_parameters() if not n.startswith('head')]
head_params     = list(base_model.head.parameters())
print(f'Backbone params : {sum(p.numel() for p in backbone_params):,}')
print(f'Head     params : {sum(p.numel() for p in head_params):,}')

# Mueve a GPU
model = base_model.to(device)

# torch.compile() acelera en Linux pero requiere Triton (no disponible en Windows)
if USE_COMPILE and hasattr(torch, 'compile'):
    model = torch.compile(model)
    print('torch.compile ✓')
else:
    print(f'torch.compile ✗  (USE_COMPILE={USE_COMPILE})')

## Sección 5 — Función de pérdida y optimizador

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    [
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': head_params,     'lr': LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

scaler = GradScaler(enabled=USE_AMP)

print('Criterion  :', criterion)
print('Optimizer  :', optimizer.__class__.__name__)
print('Scheduler  :', scheduler.__class__.__name__, f'(T_max={EPOCHS})')
print('GradScaler :', scaler._enabled)

## Sección 6 — Utilidades de checkpoint

In [ ]:
def _unwrap_model(m):
    """Desenvuelve nn.DataParallel y torch.compile (_orig_mod)."""
    if hasattr(m, 'module'):
        m = m.module
    if hasattr(m, '_orig_mod'):
        m = m._orig_mod
    return m


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, val_f1, path, is_best=False):
    state = {
        'epoch'       : epoch,
        'val_f1'      : val_f1,
        'model_state' : _unwrap_model(model).state_dict(),
        'optim_state' : optimizer.state_dict(),
        'sched_state' : scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
    }
    torch.save(state, path)
    if is_best:
        best_path = path.parent / 'best.pth'
        torch.save(state, best_path)
        print(f'  ✓ Nuevo mejor checkpoint guardado (val F1 = {val_f1:.4f})')


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    state = torch.load(path, map_location=device, weights_only=False)
    _unwrap_model(model).load_state_dict(state['model_state'])
    if optimizer and 'optim_state' in state:
        optimizer.load_state_dict(state['optim_state'])
    if scheduler and 'sched_state' in state:
        scheduler.load_state_dict(state['sched_state'])
    if scaler and 'scaler_state' in state:
        scaler.load_state_dict(state['scaler_state'])
    print(f'Checkpoint cargado: epoch={state["epoch"]}, val_f1={state["val_f1"]:.4f}')
    return state['epoch'], state['val_f1']


print('Funciones de checkpoint definidas ✓')

## Sección 7 — Resume desde checkpoint (opcional)

Ejecuta esta celda para reanudar desde el último checkpoint en `CKPT_DIR`.  
Si no existe ningún checkpoint, se omite y el entrenamiento comienza desde cero.

In [ ]:
start_epoch  = 0
best_val_f1  = 0.0
history      = []    # [{epoch, train_loss, train_f1, val_loss, val_f1, lr_backbone, lr_head, elapsed_s, is_best}]
no_improve   = 0

_latest = CKPT_DIR / 'latest.pth'
if _latest.exists():
    start_epoch, best_val_f1 = load_checkpoint(
        _latest, model, optimizer, scheduler, scaler
    )
    print(f'Reanudando desde epoch {start_epoch + 1}')
else:
    print('No hay checkpoint previo — entrenamiento desde cero.')

# Carga historial CSV si existe
_log_path = LOG_DIR / f'{MODEL_NAME}_log.csv'
if _log_path.exists():
    history = pd.read_csv(_log_path).to_dict('records')
    print(f'Historial cargado: {len(history)} epochs registradas.')

## Sección 8 — Funciones de entrenamiento y validación

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    bar = tqdm(loader, desc='  Train', leave=False)
    for images, labels in bar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            logits = model(images)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        bar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for images, labels in tqdm(loader, desc='  Val  ', leave=False):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(images)
            loss   = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1


print('train_one_epoch / validate definidas ✓')

## Sección 9 — Bucle de entrenamiento

In [ ]:
_log_path = LOG_DIR / f'{MODEL_NAME}_log.csv'

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    vl_loss, vl_f1 = validate(model, val_loader, criterion, device)
    scheduler.step()

    elapsed   = time.time() - t0
    is_best   = vl_f1 > best_val_f1
    lr_bb     = optimizer.param_groups[0]['lr']
    lr_hd     = optimizer.param_groups[1]['lr']

    if is_best:
        best_val_f1 = vl_f1
        no_improve  = 0
    else:
        no_improve += 1

    row = dict(
        epoch      = epoch + 1,
        train_loss = round(tr_loss, 6),
        train_f1   = round(tr_f1,   6),
        val_loss   = round(vl_loss, 6),
        val_f1     = round(vl_f1,   6),
        lr_backbone= lr_bb,
        lr_head    = lr_hd,
        elapsed_s  = round(elapsed, 1),
        is_best    = is_best,
    )
    history.append(row)

    # Guardar CSV incremental
    pd.DataFrame(history).to_csv(_log_path, index=False)

    # Checkpoints periódicos + latest
    save_checkpoint(model, optimizer, scheduler, scaler,
                    epoch + 1, vl_f1,
                    CKPT_DIR / 'latest.pth', is_best=is_best)

    if (epoch + 1) % 5 == 0:
        save_checkpoint(model, optimizer, scheduler, scaler,
                        epoch + 1, vl_f1,
                        CKPT_DIR / f'epoch_{epoch+1:03d}.pth')

    print(
        f'Epoch {epoch+1:>3}/{EPOCHS}  '
        f'| tr_loss={tr_loss:.4f}  tr_f1={tr_f1:.4f}'
        f'  | val_loss={vl_loss:.4f}  val_f1={vl_f1:.4f}'
        f'  | lr_bb={lr_bb:.2e}  lr_hd={lr_hd:.2e}'
        f'  | {elapsed:.0f}s'
        + ('  ← BEST' if is_best else f'  (no_improve={no_improve}/{EARLY_STOP_PAT})')
    )

    if no_improve >= EARLY_STOP_PAT:
        print(f'\nEarly stopping: {EARLY_STOP_PAT} epochs sin mejora en val F1.')
        break

print(f'\nEntrenamiento finalizado.  Mejor val F1 = {best_val_f1:.4f}')
print(f'Log guardado en: {_log_path.resolve()}')

## Sección 10 — Curvas de aprendizaje y evaluación

In [ ]:
# 10.1 Curvas de aprendizaje
df_hist = pd.read_csv(_log_path)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'{MODEL_NAME} — Curvas de Aprendizaje', fontsize=14)

ax = axes[0]
ax.plot(df_hist['epoch'], df_hist['train_loss'], label='Train')
ax.plot(df_hist['epoch'], df_hist['val_loss'],   label='Val')
best_ep = df_hist.loc[df_hist['val_f1'].idxmax(), 'epoch']
ax.axvline(best_ep, color='green', linestyle='--', alpha=0.6, label=f'Best (ep {best_ep})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Pérdida'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df_hist['epoch'], df_hist['train_f1'], label='Train')
ax.plot(df_hist['epoch'], df_hist['val_f1'],   label='Val')
ax.axvline(best_ep, color='green', linestyle='--', alpha=0.6, label=f'Best (ep {best_ep})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Macro F1')
ax.set_title('Macro F1'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best epoch : {best_ep}')
print(f'Best val F1: {df_hist["val_f1"].max():.4f}')

In [ ]:
# 10.2 Evaluación en Test — carga el mejor checkpoint
test_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'test.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

_best_ckpt = CKPT_DIR / 'best.pth'
if _best_ckpt.exists():
    load_checkpoint(_best_ckpt, model)
else:
    print('Advertencia: best.pth no encontrado — usando pesos actuales.')

model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Test'):
        images = images.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(images)
        test_preds.extend(logits.argmax(dim=1).cpu().numpy())
        test_labels.extend(labels.numpy())

test_f1  = f1_score(test_labels, test_preds, average='macro', zero_division=0)
test_acc = np.mean(np.array(test_preds) == np.array(test_labels))

print(f'\nTest Macro F1  : {test_f1:.4f}')
print(f'Test Accuracy  : {test_acc:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=CLASS_ORDER, digits=3))

In [ ]:
# 10.3 Matriz de confusión normalizada
cm = confusion_matrix(test_labels, test_preds, normalize='true')

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm, annot=True, fmt='.2%',
    xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
    cmap='Blues', ax=ax, linewidths=0.3,
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title(f'{MODEL_NAME} — Confusion Matrix (Test, normalized)', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## Sección 11 — Resumen y próximos pasos

In [ ]:
df_hist = pd.read_csv(_log_path)
best_row = df_hist.loc[df_hist['val_f1'].idxmax()]

total_s  = df_hist['elapsed_s'].sum()
h, rem   = divmod(int(total_s), 3600)
m, s     = divmod(rem, 60)

print('=' * 60)
print(f'  Modelo          : {MODEL_NAME}')
print(f'  Epochs          : {int(df_hist["epoch"].max())} / {EPOCHS}')
print(f'  Best epoch      : {int(best_row["epoch"])}')
print(f'  Best val F1     : {best_row["val_f1"]:.4f}')
print(f'  Test Macro F1   : {test_f1:.4f}')
print(f'  Test Accuracy   : {test_acc:.4f}')
print(f'  Tiempo total    : {h}h {m:02d}min {s:02d}s')
print(f'  Checkpoints     : {CKPT_DIR.resolve()}')
print(f'  Log             : {_log_path.resolve()}')
print('=' * 60)
print()
print('Comparativa acumulada:')
print('  04  EfficientNet-B3 → best val F1 = 0.6894  (epoch 16/30)')
print('  05  ResNet-50       → best val F1 = 0.6914  (epoch 14/19)')
print(f'  06  Swin-S          → best val F1 = {best_row["val_f1"]:.4f}  (epoch {int(best_row["epoch"])}/{int(df_hist["epoch"].max())})')
print()
print('Próximo paso → 07_train_maxvit_local.ipynb  (MaxViT-T, IMAGE_SIZE=224)')